# Model v14: v12 + parametri fisici e costi di avvio/arresto

v14 parte da v12 (variante N) e aggiunge la variante **P**:

- **Fisica per unità** (dallo YAML): salto netto con perdite, rendimento al picco, potenza marginale della n-esima unità accesa → `rel` = quanta energia rende la stessa acqua rispetto al nominale
- **Costi di avvio/arresto**: `viterbi` calcola per ogni ora il guadagno di stare ON invece che OFF, tenendo conto di start + stop (conta soprattutto per Haukeli, Vesle Kjela, Hogga, Byrte)
- **Budget d'acqua nella soglia**: `lam` = water value finale, alzato se l'acqua disponibile non copre le ore e abbassato se serve evitare lo sfioro
- **Apporti a monte mancanti** nel budget massimo: Staavatn → Vesle Kjela, Langesae → Kjela, Venemo + Hyljelihyl → Vinje

Selezione per centrale sul test locale come v12 (score della scelta ottimistico: confrontare soprattutto N e P pure). `RW = 2.0` di default nella submission.

### getting the data (+ building the functions and class)

In [1]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from functools import cached_property
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view as swv
import yaml
import xgboost as xgb

In [2]:
# create the classes and funcs to prepare the data
def build_cases(X, Y, T=168):
    '''
    Modify the dataset to have them case indexed
    Input:
    - X: x dataset
    - Y: y dataset
    '''
    hourly = ['day_ahead_price'] + [c for c in X.columns if c.endswith('_inflow')]
    vol = [c for c in X.columns if c.endswith('_volume')]
    wv  = [c for c in X.columns if c.endswith('_wv')]

    # getting the position of every case starting hour
    pos = X.index.get_indexer(Y.index)
    assert (pos >= 0).all() and (pos + T < len(X)).all()

    # creating the sliding windows
    win = swv(X[hourly].to_numpy(np.float32), T, axis=0)
    xh = win[pos].reshape(len(pos), -1)

    wv_arr = X[wv].to_numpy()
    xs = np.hstack([
        X[vol].to_numpy()[pos],  # initial volume (s)
        wv_arr[pos],             # initial water value (pos)
        wv_arr[pos + T]          # final water value (pos+T)
    ])

    hourly_cols = [f"{c}_t{h}" for c in hourly for h in range(T)]
    all_cols = hourly_cols + vol + [f"{c}_init" for c in wv] + [f"{c}_final" for c in wv]

    # excluding run_no e total_calculation_time
    y = pd.DataFrame(Y.filter(like='result_committed').to_numpy(np.uint8), index=Y.index, columns=Y.columns[2:])
    x = pd.DataFrame(np.hstack([xh, xs]).astype(np.float32), index=Y.index, columns=all_cols)
    return x, y


class DataLoader:
    def __init__(self, data_source):
        self.data_source = Path(data_source)

    def _csv(self, folder, filename, date_columns=()):
        frame = pd.read_csv(self.data_source / folder / filename)
        for column in date_columns:
            if column in frame.columns:
                frame[column] = pd.to_datetime(frame[column])
        return frame

    def _yaml(self, folder, filename):
        with (self.data_source / folder / filename).open(encoding="utf-8") as stream:
            return yaml.safe_load(stream)

    @cached_property
    def day_ahead_price_data(self):
        return self._csv("kernel", "Historical_day_ahead_price_2015_2025.csv", ("date",))

    @cached_property
    def inflow_data(self):
        return self._csv("kernel", "Historical_inflow_1958_2025.csv", ("date",))

    @cached_property
    def volume_data(self):
        return self._csv("kernel", "Historical_volume_2015_2024.csv", ("date",))

    @cached_property
    def synthetic_water_value_data(self):
        return self._csv("kernel", "Synthetic_water_value_2015_2024.csv", ("date",))

    @cached_property
    def commitment_decision_data(self):
        return self._csv("kernel", "Unit_commitment_decisions.csv", ("starttime",))

    @cached_property
    def min_flow_constraint_data(self):
        return self._csv("extended", "Constraint_min_flow.csv", ("date",))

    @cached_property
    def min_volume_constraint_data(self):
        return self._csv("extended", "Constraint_min_volume.csv", ("date",))

    @cached_property
    def topology_data(self):
        return self._yaml("extended", "Tokke_Vinje_topology.yaml")


# Class to create the entire dataset in one go
class Training_set(DataLoader):
    def __init__(self, data_source):
        super().__init__(data_source)
        self.y = None
        self.x = None

    def load_copies(self):
        return {
            "day_ahead_price": self.day_ahead_price_data,
            "inflow": self.inflow_data,
            "volume": self.volume_data,
            "wv": self.synthetic_water_value_data,
            "uc": self.commitment_decision_data,
        }

    def create_x(self):
        data = self.load_copies()

        # setting the date as index and fixing the columns names
        for name in data.keys():
            if 'date' in list(data[name].columns):
                data[name]['date'] = pd.to_datetime(data[name]['date'], utc=True)
                data[name] = data[name].set_index(data[name]['date']).drop(columns='date')
                if 'Day-ahead price (EUR/MWh)' in data[name].columns:
                    data[name].columns = data[name].columns.str.replace(r'\s*\([^)]*\)', '', regex=True).str.replace(r'[\s-]+', '_', regex=True).str.lower()
                else:
                    data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')
            else:
                data[name]['starttime'] = pd.to_datetime(data[name]['starttime'], utc=True)
                data[name] = data[name].set_index(data[name]['starttime']).drop(columns='starttime')
                data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')

        # wv and volume have daily data: repeat their value hourly
        hourly = data['day_ahead_price'].index
        for name in ('volume', 'wv'):
            data[name] = data[name].reindex(hourly, method='ffill')

        self.y = data.pop('uc').copy()
        x = data.pop('day_ahead_price')
        for name in data.keys():
            x = x.join(data[name].add_suffix('_' + name), how='left')
        self.x = x.copy()

In [3]:
DATA = './Users/lucaag/data'
ts = Training_set(data_source=DATA)
ts.create_x()
X_all, Y_all = ts.x.sort_index(), ts.y.sort_index()

split = int(len(Y_all) * 0.8)  # stessa split di v4
x_train, y_train = build_cases(X_all, Y_all.iloc[:split])
x_test, y_test = build_cases(X_all, Y_all.iloc[split + 7:])
topo = ts.topology_data['model']
# vincoli: 6 min_flow + 3 min_volume, serie orarie 2015-2025 (come v6)
C = pd.concat([ts.min_flow_constraint_data.set_index('date'), ts.min_volume_constraint_data.set_index('date')], axis=1).astype(np.float32)
C.index = pd.to_datetime(C.index, utc=True)
for x in (x_train, x_test): assert (x.index.to_series().diff().dropna() == pd.Timedelta('1D')).all()  # casi consecutivi
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

(2337, 3243) (2337, 2352) (578, 3243) (578, 2352)


## Topology and target

In [4]:
T = 168

# physical constraints from the topological yaml file
RES = {k.lower(): v for k, v in topo['reservoir'].items()}
PL = {k.lower().replace(' ', '_'): v for k, v in topo['plant'].items()}
GEN = {k.lower().replace(' ', '_'): g for k, g in topo['generator'].items()}
PN = {k: g['p_nom'] for k, g in GEN.items()}

# every power plant to its upstream source reservoirs and its downstream receiving reservoir
TOPO = {
    'vesle_kjela': (['kjelavatn'], 'foersvatn'),
    'kjela': (['foersvatn', 'bordalsvatn'], 'hyljelihyl'),
    'songa': (['songavatn', 'bitdalsvatn'], 'totak'),
    'vinje': (['vaamarvatn', 'totak'], 'vinjevatn'),
    'haukeli': (['langeidvatn', 'vatjern'], 'vinjevatn'),  # Langeidvatn primo (volume Vatjern costante nei dati)
    'tokke': (['vinjevatn'], 'bandak'),
    'byrte': (['botnedalsvatn'], 'byrtevatn'),
    'lio': (['byrtevatn'], 'bandak'),
    'hogga': (['bandak'], None),
}

# linking the downstream plants to their upstream counterparts
UP_PLANTS = {'kjela': ['vesle_kjela'], 'vinje': ['kjela', 'songa'], 'tokke': ['vinje', 'haukeli'],
             'hogga': ['tokke', 'lio'], 'lio': ['byrte']}
SRC = {'vinje': 'totak'}
FEED = {'vesle_kjela': ['staavatn'], 'kjela': ['langesae'], 'vinje': ['venemo', 'hyljelihyl']}  # v14: bacini che alimentano i serbatoi a monte

# upstream plants first: their predicted probabilities become features of the downstream plants
ORDER = ['vesle_kjela', 'songa', 'byrte', 'haukeli', 'kjela', 'lio', 'vinje', 'tokke', 'hogga']

units = sorted({c[len('result_committed_'):].rsplit('_t', 1)[0] for c in y_train.columns})
plant_units = {p: [i for i, u in enumerate(units) if u.rsplit('_g', 1)[0] == p] for p in ORDER}
ucols = lambda u: [f'result_committed_{units[u]}_t{h}' for h in range(T)]
stack3 = lambda y: np.stack([y[ucols(u)].to_numpy() for u in range(len(units))], axis=1)  # (casi, unità, 168)
Y3, Y3te = stack3(y_train), stack3(y_test)
Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]
out_cols = [c for u in range(len(units)) for c in ucols(u)]
auc = lambda Y, P: roc_auc_score(Y.ravel(), P.ravel())  # micro AUC

## Feature v4

In [5]:
PRICE = [f'day_ahead_price_t{h}' for h in range(T)]

def shift(a, k):
    o = np.roll(a, k, 1)
    if k > 0: o[:, :k] = a[:, :1]
    else: o[:, k:] = a[:, -1:]
    return o

def head(r, v):
    d = RES[r]['vol_head']
    return np.interp(v, d['x'], d['y'])

lw = lambda a: np.log(np.clip(a, 1, None))  # log senza scala (prezzi/wv < 1 -> 0)

def static(X, plant, logwv=False):
    up, dn = TOPO[plant]; rs = up + ([dn] if dn else [])
    cols = [f'{r}_{s}' for r in rs for s in ('volume', 'wv_init', 'wv_final')]
    A0 = X[cols].to_numpy()
    if logwv: A0 = np.where(np.array([not c.endswith('_volume') for c in cols]), lw(A0), A0)
    S = [A0]
    S += [(X[f'{r}_volume'] / RES[r]['max_vol']).to_numpy()[:, None] for r in rs]
    hu = head(up[0], X[f'{up[0]}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else np.zeros_like(hu)
    wu = X[f'{up[0]}_wv_final'].to_numpy()
    wd = X[f'{dn}_wv_final'].to_numpy() if dn else np.zeros(len(X))
    S.append(np.column_stack([hu, hu - np.maximum(hd, PL[plant]['outlet_line']),
                              lw(wu) - lw(wd) if logwv else wu - wd]))
    return np.hstack(S)

def inflow(X, up):  # inflow orario sommato sui bacini a monte (m3/s)
    return sum((X[[f'{r}_inflow_t{h}' for h in range(T)]].to_numpy() for r in up if f'{r}_inflow_t0' in X.columns),
               np.zeros((len(X), T), np.float32))

## optimizer surrogate

In [6]:
ETA = 0.88
CAP_RES = {'haukeli': 'vatjern'}

def cap(X, plant):  # portata massima di centrale (Mm3/h)
    up, dn = TOPO[plant]
    r0 = CAP_RES.get(plant, up[0])  # Haukeli prende l'acqua da Vatjern
    hu = head(r0, X[f'{r0}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else 0
    H = np.maximum(hu - np.maximum(hd, PL[plant]['outlet_line']), 5)
    return sum(PN[units[u]] for u in plant_units[plant]) / (2.725 * ETA * H)

CC, _cw = list(C.columns), {}
def cwin(X):
    k = (X.index[0], len(X))
    if k not in _cw:
        pos = C.index.get_indexer(X.index); assert (pos >= 0).all()
        _cw[k] = swv(C.to_numpy(), T, axis=0)[pos]
    return _cw[k]
mvol = lambda X, rs: sum((cwin(X)[:, CC.index(r.capitalize())].max(1) for r in rs if r.capitalize() in CC), np.zeros(len(X), np.float32))
CONS = {'lio': ['r_Vest_Vassdraget', 'b_Byrtevatn', 'b_Vinjevatn', 'Byrtevatn'], 'byrte': ['b_Byrtevatn', 'Byrtevatn'], 'songa': ['Totak'],
        'vinje': ['Totak', 'b_Vinjevatn'], 'vesle_kjela': ['b_Kjelavatn'], 'kjela': ['b_Hyljelihyl'], 'tokke': ['b_Vinjevatn'], 'hogga': ['b_Bandak']}
wsrc = lambda plant, cons: {**SRC, **({'haukeli': 'langeidvatn'} if cons else {})}.get(plant, TOPO[plant][0][0])

def surr(X, plant, rel=None, cons=False):
    up, c, nu = TOPO[plant][0], cap(X, plant), len(plant_units[plant])
    src = wsrc(plant, cons)
    V0 = sum(X[f'{r}_volume'].to_numpy() for r in up)
    Win = inflow(X, up).sum(1) * 0.0036 + (rel.sum(1) if rel is not None else 0)  # m3/s * 0.0036 = Mm3/h
    Wmax = np.clip(V0 + Win - (mvol(X, up) if cons else 0), 0, T * c)  # massimo utilizzabile
    Wmin = np.minimum(np.maximum(V0 + Win - sum(RES[r]['max_vol'] for r in up), 0), Wmax)  # minimo per non sfiorare
    P = X[PRICE].to_numpy(); rank = (-P).argsort(1).argsort(1)
    Wp = [np.clip((P > X[f'{src}_{w}'].to_numpy()[:, None]).sum(1) * c, Wmin, Wmax) for w in ('wv_init', 'wv_final')]
    Ws = [Wmin, *Wp, Wmax]
    S = [np.clip((W / c)[:, None] - rank, 0, 1) * nu for W in Ws]
    S += [np.repeat((W / c / T)[:, None], T, 1) for W in Ws]
    if rel is not None:
        S += [rel / c[:, None] * nu, np.cumsum(rel, 1) / c[:, None]]
    return S

def cons_feats(X, plant, c):
    up, dn = TOPO[plant]; n, W = len(X), cwin(X)
    bc = lambda g: np.repeat(np.asarray(g, np.float32)[:, None], T, 1)
    Vd = X[f'{dn}_volume'].to_numpy() if dn else np.zeros(n)
    Id = inflow(X, [dn]).sum(1) * 0.0036 if dn else np.zeros(n)
    F = [W[:, CC.index(k)] for k in CONS.get(plant, [])]
    F += [bc(sum(X[f'{r}_volume'].to_numpy() for r in up) - mvol(X, up)),
          bc(np.maximum(mvol(X, [dn] if dn else []) - Vd - Id, 0) / c)]
    if plant == 'lio':
        assert 'r_vest_vassdraget_inflow_t0' in X.columns
        w = lambda k: W[:, CC.index(k)]
        need = np.maximum(w('r_Vest_Vassdraget') - inflow(X, ['r_vest_vassdraget']) - w('b_Vinjevatn') - w('b_Byrtevatn'), 0)
        q = (c / 0.0036)[:, None]  # portata massima (m3/s)
        F += [need, np.cumsum(need, 1) / q, bc(need.sum(1) / q[:, 0])]
    return F

## v14: fisica e costi di avvio

Per ogni unità, nell'ordine di accensione: `viterbi` restituisce (miglior piano con unità ON all'ora t) − (miglior piano con unità OFF), al netto di start + stop. Positivo = conviene essere accesa.

In [7]:
NF = 7  # feature per unità

def viterbi(m, s):  # m: margine orario di stare ON (casi, T); s: costo di un ciclo start + stop
    h = s / 2  # metà costo per ogni cambio di stato (stessa somma su un ciclo)
    Fon, Fof, Bon, Bof = m.copy(), np.zeros_like(m), np.zeros_like(m), np.zeros_like(m)
    for t in range(1, T):
        Fon[:, t] = m[:, t] + np.maximum(Fon[:, t-1], Fof[:, t-1] - h)
        Fof[:, t] = np.maximum(Fof[:, t-1], Fon[:, t-1] - h)
    for t in range(T - 2, -1, -1):
        Bon[:, t] = np.maximum(m[:, t+1] + Bon[:, t+1], Bof[:, t+1] - h)
        Bof[:, t] = np.maximum(Bof[:, t+1], m[:, t+1] + Bon[:, t+1] - h)
    return Fon + Bon - Fof - Bof

def peak(g):  # portata, salti di riferimento e rendimenti al picco delle curve della turbina
    C_ = g['turb_eff_curves']; b = [int(np.argmax(c['y'])) for c in C_]
    return C_[0]['x'][b[0]], np.array([c['ref'] for c in C_], float), np.array([c['y'][i] / 100 for c, i in zip(C_, b)])

def unit_feats(X, plant, hmin, hmax):
    up, dn = TOPO[plant]; pl = PL[plant]; r0 = CAP_RES.get(plant, up[0])
    P = X[PRICE].to_numpy(); w = X[f'{wsrc(plant, True)}_wv_final'].to_numpy()
    Hg = head(r0, X[f'{r0}_volume'].to_numpy()) - np.maximum(head(dn, X[f'{dn}_volume'].to_numpy()) if dn else 0, pl['outlet_line'])
    # soglia lam: water value, spostato solo se il budget d'acqua lo impone
    Ps = -np.sort(-P, 1); pk = lambda h: np.take_along_axis(Ps, np.clip(h.astype(int), 0, T - 1)[:, None], 1)[:, 0]
    lam = np.clip(w, np.where(hmax < T, pk(hmax), -np.inf), np.where(hmin > 0, pk(hmin), np.inf))
    w, lam = w[:, None], lam[:, None]
    wn, ln = np.maximum(w, 0) + 5, np.maximum(lam, 0) + 5
    F, prev = [], 0
    for j, u in enumerate(plant_units[plant]):
        g = GEN[units[u]]; q, refs, et = peak(g)
        Hj = np.maximum(Hg - float(pl['main_loss'][0]) * ((j + 1) * q)**2 - pl['penstock_loss'][g['penstock'] - 1] * q**2, 1)
        tot = 9.81e-3 * (j + 1) * q * Hj * np.interp(Hj, refs, et) * .98
        pm = np.minimum(tot - prev, g['p_max']); prev = tot  # potenza marginale della (j+1)-esima unità
        pref = 9.81e-3 * q * refs.max() * et.max() * .98
        rel, s = (pm / pref)[:, None], (g['startcost_const'] + g['stopcost_const']) / pref  # EUR per MW di riferimento
        D1, D2, D3 = viterbi(P - w, s), viterbi(rel * P - w, s), viterbi(rel * P - lam, s)
        F += [D1 / wn, D2 / wn, D3 / ln, D1 > 0, D3 > 0, np.repeat(rel, T, 1), np.repeat(lw(lam) - lw(w), T, 1)]
    return F

## feature di centrale

In [8]:
def base(X, plant, up_prob=None, rel=None, var='N'):
    up, dn, n = TOPO[plant][0], TOPO[plant][1], len(X)
    P = X[PRICE].to_numpy(); D = P.reshape(n, 7, 24)
    I = inflow(X, up)
    wvf = X[f'{up[0]}_wv_final'].to_numpy()[:, None]
    h = np.tile(np.arange(T), (n, 1))
    rank, drank = P.argsort(1).argsort(1) / (T - 1), D.argsort(2).argsort(2).reshape(n, T) / 23
    dmean, wmean = np.repeat(D.mean(2), 24, 1), P.mean(1, keepdims=True)
    rl, sd = lw(P) - lw(wmean), P.std(1, keepdims=True) + 1  # senza scala: livelli relativi, z-score, log(P/wv)
    PF = [rl, shift(rl, 1), shift(rl, -1), rank, drank, (P - dmean) / sd, (P - wmean) / sd, lw(P) - lw(wvf)]
    F = [h, h % 24, (X.index.dayofweek.to_numpy()[:, None] + h // 24) % 7] + PF + [I, np.repeat(I.mean(1, keepdims=True), T, 1)]
    if up_prob is not None:
        F += [up_prob, shift(up_prob, 1), shift(up_prob, 3), np.repeat(up_prob.mean(1, keepdims=True), T, 1)]
    dif = lambda r, w: lw(P) - lw(X[f'{r}_wv_{w}'].to_numpy()[:, None])
    F += cons_feats(X, plant, cap(X, plant)) + [dif(r, w) for r in up + ([dn] if dn else []) for w in ('init', 'final')]
    c = cap(X, plant)[:, None]
    ci = np.cumsum(I, 1) * 0.0036
    V0 = sum(X[f'{r}_volume'].to_numpy() for r in up)[:, None]
    F += [ci / c, (V0 + ci - sum(RES[r]['max_vol'] for r in up)) / c]  # inflow cumulato e pressione di sfioro
    S = surr(X, plant, rel, True)
    F += S
    A = [np.repeat(static(X, plant, True), T, 0), np.repeat(X.index.dayofyear.to_numpy(), T)]
    if var == 'P':
        fr = FEED.get(plant, [])
        feed = sum((X[f'{r}_volume'].to_numpy() + inflow(X, [r]).sum(1) * 0.0036 for r in fr), np.zeros(n)) / c[:, 0]
        U = unit_feats(X, plant, S[4][:, 0] * T, S[7][:, 0] * T + feed)  # S[4], S[7]: Wmin e Wmax in frazione di settimana
        A.append(np.repeat(feed, T))
    else: U = []
    # colonne: comuni | NF per unità | S[2] (ultima, usata da design)
    return np.column_stack([f.ravel() for f in F] + A + [f.ravel() for f in U] + [S[2].ravel()]).astype(np.float32)

# training

Come v12: XGBoost per centrale con unità impilate, CV a fold contigui con gap, stacking topologico. In `design` ogni unità riceve solo le proprie `NF` feature.

In [9]:
NE, FOLDS, GAP, SEEDS = 200, 5, 7, 3

def mk(seed):
    return xgb.XGBClassifier(n_estimators=NE, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             tree_method='hist', n_jobs=-1, random_state=seed)

def fp(X, y, Xs, w=None):
    if y.min() == y.max():
        return [np.full(len(x), float(np.clip(y[0], .01, .99))) for x in Xs]
    ms = [mk(s).fit(X, y, sample_weight=w) for s in range(SEEDS)]
    return [np.mean([m.predict_proba(x)[:, 1] for m in ms], 0) for x in Xs]

def folds(n):
    b = np.linspace(0, n, FOLDS + 1, dtype=int)
    for a, c in zip(b[:-1], b[1:]):
        yield np.r_[0:max(a - GAP, 0), min(c + GAP, n):n], np.arange(a, c)

ridx = lambda c: (np.asarray(c)[:, None] * T + np.arange(T)).ravel()
ylong = lambda c, us: np.concatenate([Yflat[u][ridx(c)] for u in us])

def design(B, c, us, var):
    r = ridx(c); R = B[r]
    k = NF if var == 'P' else 0; s0 = R.shape[1] - 1 - k * len(us)  # inizio delle feature per unità
    def blk(j, u):
        return np.column_stack([R[:, :s0], R[:, s0 + j * k: s0 + (j + 1) * k], np.full(len(r), u), np.full(len(r), j), R[:, -1] - j])
    return np.vstack([blk(j, u) for j, u in enumerate(us)])

def fit_group(B, Bt, us, var, cw=None):
    n, nt = len(B) // T, len(Bt) // T
    allc = np.arange(n)
    wl = lambda c: None if cw is None else np.tile(np.repeat(cw[c], T), len(us))  # peso per caso -> righe
    te = fp(design(B, allc, us, var), ylong(allc, us), [design(Bt, np.arange(nt), us, var)], wl(allc))[0].reshape(len(us), nt, T)
    oo = np.zeros((len(us), n, T))
    for tr, va in folds(n):
        p = fp(design(B, tr, us, var), ylong(tr, us), [design(B, va, us, var)], wl(tr))[0]
        oo[:, va] = p.reshape(len(us), len(va), T)
    return oo, te

def run(VAR, xtr, xte, rw=1.0):
    cw = None if rw == 1 else np.where(xtr.index.year >= 2021, rw, 1.0).astype(np.float32)
    oof, te, rtr, rte = {}, {}, {}, {}
    P = np.zeros((len(xte), len(units), T), np.float32); O = np.zeros((len(xtr), len(units), T), np.float32)
    for plant in ORDER:
        us, ups, v = plant_units[plant], UP_PLANTS.get(plant, []), VAR[plant]
        m = lambda d: np.mean([d[p] for p in ups], 0) if ups else None
        s = lambda d: sum(d[p] for p in ups) if ups else None
        oo, t = fit_group(base(xtr, plant, m(oof), s(rtr), v), base(xte, plant, m(te), s(rte), v), us, v, cw)
        P[:, us, :], O[:, us, :] = t.transpose(1, 0, 2), oo.transpose(1, 0, 2)
        te[plant], oof[plant] = t.mean(0), oo.mean(0)
        rtr[plant], rte[plant] = oof[plant] * cap(xtr, plant)[:, None], te[plant] * cap(xte, plant)[:, None]
        print(plant, 'ok')
    return P, O

## comparison

In [10]:
VARS = ('N', 'P')
preds = {v: run({p: v for p in ORDER}, x_train, x_test) for v in VARS}
oo = pd.DataFrame({v: {p: auc(Y3[:, u], preds[v][1][:, u]) for p, u in plant_units.items()} for v in VARS})
tt = pd.DataFrame({v: {p: auc(Y3te[:, u], preds[v][0][:, u]) for p, u in plant_units.items()} for v in VARS})
SEL = tt  # come v12: test locale (score della scelta ottimistico); usare oo per selezionare su OOF
VAR = {p: next(v for v in VARS if SEL.loc[p, v] >= SEL.loc[p].max() - 5e-4) for p in ORDER}
P, O = np.zeros_like(preds['N'][0]), np.zeros_like(preds['N'][1])
for p, u in plant_units.items(): P[:, u], O[:, u] = preds[VAR[p]][0][:, u], preds[VAR[p]][1][:, u]
print('variante scelta:', VAR)
for v in VARS: print(v, f'oof {auc(Y3, preds[v][1]):.4f} test {auc(Y3te, preds[v][0]):.4f}')
print(f'scelta per centrale: oof {auc(Y3, O):.4f} test {auc(Y3te, P):.4f}   (v12 N: oof 0.9780 test 0.9806)')
pd.concat({'oof': oo, 'test': tt}, axis=1).round(4)

vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
variante scelta: {'vesle_kjela': 'N', 'songa': 'P', 'byrte': 'P', 'haukeli': 'N', 'kjela': 'P', 'lio': 'P', 'vinje': 'N', 'tokke': 'P', 'hogga': 'N'}
N oof 0.9780 test 0.9807
P oof 0.9788 test 0.9823
scelta per centrale: oof 0.9786 test 0.9820   (v12 N: oof 0.9780 test 0.9806)


oof            test        
                  N       P       N       P
vesle_kjela  0.9916  0.9932  0.9938  0.9941
songa        0.9741  0.9766  0.9742  0.9874
byrte        0.9949  0.9953  0.9954  0.9983
haukeli      0.8760  0.8737  0.9002  0.8983
kjela        0.9603  0.9629  0.9783  0.9801
lio          0.9645  0.9674  0.9325  0.9397
vinje        0.9770  0.9774  0.9835  0.9836
tokke        0.9888  0.9886  0.9895  0.9904
hogga        0.9700  0.9705  0.9791  0.9792

In [11]:
pd.DataFrame({v: [auc(Y3te[:, i], preds[v][0][:, i]) for i in range(len(units))] for v in VARS}, index=units).round(4)

,N,P
byrte_g1,0.9954,0.9983
haukeli_g1,0.9002,0.8983
hogga_g1,0.9791,0.9792
kjela_g1,0.9783,0.9801
lio_g1,0.9325,0.9397
songa_g1,0.9742,0.9874
tokke_g1,0.9908,0.9914
tokke_g2,0.9867,0.9877
tokke_g3,0.9914,0.9918
tokke_g4,0.9885,0.9896


## kaggle submission

In [13]:
MAP, SS = f'{DATA}/prediction_mapping.csv', f'{DATA}/sample_submission.csv'  # adattare ai percorsi reali
mp = pd.read_csv(MAP); mp['starttime'] = pd.to_datetime(mp['starttime'], utc=True)
dummy = pd.DataFrame(0, index=pd.DatetimeIndex(mp['starttime']), columns=['Run No', 'total_calculation_time', *y_train.columns])
x_all, y_all = build_cases(X_all, Y_all)
x_kag, _ = build_cases(X_all, dummy)
assert len(x_all) == 2922 and len(x_kag) == len(mp)
Y3 = stack3(y_all); Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]  # run() usa questi target globali
RW = 2.0  # peso casi 2021+ (il migliore in v12)
VAR = {p: 'P' for p in ORDER}
Pk, _ = run(VAR, x_all, x_kag, RW)
sub = pd.DataFrame(Pk.reshape(len(Pk), -1), columns=out_cols)[list(y_train.columns)]
sub.insert(0, 'Run No', mp['Run No'].to_numpy())
ss = pd.read_csv(SS); norm = lambda c: c.lower().replace(' ', '_'); nm = {norm(c): c for c in ss.columns}
sub = sub.rename(columns=lambda c: nm.get(norm(c), c))  # nomi colonna come in sample_submission.csv
assert set(sub.columns) == set(ss.columns), set(sub.columns) ^ set(ss.columns)
sub = sub.set_index('Run No').loc[ss['Run No']].reset_index()[list(ss.columns)]
sub.to_csv(f'{DATA}/submission_v14_allP.csv', index=False)
print(sub.shape, sub.iloc[:, 1:].to_numpy().mean().round(3))

vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
(725, 2353) 0.464
